# 💾 Checkpoints

## What are Checkpoints?
Checkpoints **save graph state** at each step.

This enables:
- **Resume** interrupted workflows
- **Time travel** — go back to any previous state
- **Human-in-the-loop** — pause and wait for human input
- **Debugging** — inspect state at any step

## Checkpoint Backends
| Backend | Storage | Use Case |
|---------|---------|---------|
| MemorySaver | RAM | Development |
| SqliteSaver | SQLite file | Small production |
| PostgresSaver | PostgreSQL | Large production |
| RedisSaver | Redis | High throughput |


In [ ]:
# ── Checkpointing with MemorySaver ───────────────────────────────────────
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI(model='gpt-4o-mini')

def chat_node(state: ChatState) -> dict:
    system = SystemMessage(content='You are a helpful assistant.')
    response = llm.invoke([system] + state['messages'])
    return {'messages': [response]}

# Create checkpointer
checkpointer = MemorySaver()

graph = StateGraph(ChatState)
graph.add_node('chat', chat_node)
graph.set_entry_point('chat')
graph.add_edge('chat', END)

# Compile WITH checkpointer
app = graph.compile(checkpointer=checkpointer)

# thread_id = unique conversation ID
config = {'configurable': {'thread_id': 'user_alice_001'}}

# Turn 1
r1 = app.invoke({'messages': [HumanMessage('My name is Alice')]}, config=config)
print('Turn 1:', r1['messages'][-1].content)

# Turn 2 — state is automatically resumed!
r2 = app.invoke({'messages': [HumanMessage('What is my name?')]}, config=config)
print('Turn 2:', r2['messages'][-1].content)

# Inspect state at any point
state = app.get_state(config)
print(f'\nTotal messages in checkpoint: {len(state.values["messages"])}')

## 🎯 Interview Questions

1. **What is a checkpointer in LangGraph?**
2. **What is thread_id and why is it important?**
3. **What checkpoint backends are available?**
4. **How do you implement time travel in LangGraph?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Checkpoints — Saving and Resuming State**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
